In [1]:
#A model learns from one part of the data and is tested on another it has never seen. Preprocessing sets this up correctly. In this step we make sure that no data from the test set is used to train the model. This is important because we want to evaluate how well our model generalizes to new data, not how well it memorizes the training data.

In [2]:
#LOADING THE NEEDED LIBRARIES
import pandas as pd, numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

In [3]:
#LOADING THE DATA AND SEPARATING THE INPUT VALUES FROM THE ANSWER. HERE X=> INPUT VALUES AND Y=> OUTPUT VALUES
import os

df = pd.read_csv("../data/raw/creditcard.csv")
print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates()
print("Rows after removing duplicates:", len(df))
X = df.drop(columns="Class")   #INPUTS FOR THE MODEL
y = df["Class"]                #0 - NORMAL, 1 - FRAUD
print(X.shape, y.shape)

Rows before removing duplicates: 284807
Rows after removing duplicates: 283726
(283726, 30) (283726,)


In [ ]:
#SPLITTING THE DATASET INTO TRAINING, TESTING AND VALIDATION SETS
#WE PUT 60% OF THE DATA INTO THE TRAINING SET, 20% INTO THE VALIDATION SET AND 20% INTO THE TEST SET. WE ALSO STRATIFY THE DATA TO ENSURE THAT THE FRAUD RATE IS SIMILAR IN ALL THREE SETS.
#RANDOM STATE MAKES THE SPLIT IDENTICAL EVERY TIME CODE IS RUN
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)
print("Fraud rate  train / val / test:",
      y_train.mean(), y_val.mean(), y_test.mean())

Train: (170235, 30)  Val: (56745, 30)  Test: (56746, 30)
Fraud rate  train / val / test: 0.0016682820806532147 0.0016565336152965019 0.0016741268107003137


In [6]:
#SCALING AMOUNT AND TIME. WITHOUT SCALING, THE MODELS WOULD LET BIG NUMBERS DOMINATE THE SMALLER ONES.
#ROBUSTSCALER RESCALES THE DATA USING MEDIAN.
#JOBLIB IS USED TO SAVE THE SCALER OBJECT SO THAT IT CAN BE USED LATER TO SCALE NEW DATA BEFORE MAKING PREDICTIONS.cols = ["Amount", "Time"]
cols = ["Amount", "Time"]
scaler = RobustScaler().fit(X_train[cols])     #LEARNING THE RULE FROM TRAINING DATA ONLY.

def scale(d):
    d = d.copy()
    d[cols] = scaler.transform(d[cols])
    return d

X_train = scale(X_train)
X_val = scale(X_val)
X_test = scale(X_test)

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.joblib")

X_train[cols].describe()

,Amount,Time
count,170235.000000,170235.000000
mean,0.920454,0.117763
std,3.386083,0.557738
min,-0.305407,-0.997364
25%,-0.227459,-0.359752
50%,0.000000,0.000000
75%,0.772541,0.640248
max,262.205872,1.031199


In [7]:
os.makedirs("data/processed", exist_ok=True)

for name, d in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test),
                ("y_train", y_train), ("y_val", y_val), ("y_test", y_test)]:
    d.to_csv(f"data/processed/{name}.csv", index=False)

print(os.listdir("data/processed"))

['X_test.csv', 'X_train.csv', 'X_val.csv', 'y_test.csv', 'y_train.csv', 'y_val.csv']
